# 01 Quickstart: train a NICHEVERSE codebook

**NICHEVERSE** learns two coupled vector-quantized codebooks from imaging spatial transcriptomics: a **cell codebook** (recurrent transcriptional states) and a **neighborhood codebook** (recurrent multicellular niches), coupled by cross-attention.

This notebook trains an end-to-end model on a **real** MERFISH mouse retina dataset (Vizgen, 4 samples), then reads back the codes and summarizes them.

The three inputs NICHEVERSE needs from an AnnData:

1. raw counts in `adata.X`
2. micron coordinates in `adata.obsm['spatial']`
3. a sample column `adata.obs['sample_id']` (the neighbor graph is built within each sample only)

**Demo note:** we use ~25 epochs so this runs in a few minutes; a real cohort run uses `num_epochs ~300`. The recommended default encoder is `mlp_deep` (a SwiGLU pre-norm residual MLP that gives the healthiest raw codebook on sparse imaging counts).

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import scanpy as sc
import nicheverse as nv
from nicheverse import ModelConfig, TrainConfig
# bundled example data lives next to the notebooks, in ../examples/data
DATA = os.path.join('..', 'examples', 'data')
print('nicheverse', nv.__version__)

/data1/massaguj/yarlagad/conda/envs/annotforimst/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


/data1/massaguj/yarlagad/conda/envs/annotforimst/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


nicheverse 0.2.0


## Load the data

We use the bundled **real** MERFISH mouse retina dataset (Vizgen, 4 samples). `read_spatial` standardizes any AnnData or `.h5ad` path: it guarantees `obsm['spatial']` and the sample column exist. `adata.X` here is raw counts; `train_model` applies normalize + log1p internally.

In [2]:
adata = nv.read_spatial(os.path.join(DATA, 'merfish_retina.h5ad'),
                        sample_col='sample_id', spatial_key='spatial')
print(adata)
print('samples:', list(adata.obs["sample_id"].astype(str).unique()))
print('cells:', adata.n_obs, ' genes:', adata.n_vars)

AnnData object with n_obs × n_vars = 113385 × 368
    obs: 'sample_id'
    obsm: 'spatial'
samples: ['VZG105a_WT1', 'VZG105a_WT2', 'VZG105a_WT3', 'VZG105a_WT4']
cells: 113385  genes: 368


## Configure and train

`ModelConfig` describes the architecture (encoder, codebook sizes). `TrainConfig` describes the optimization and the spatial neighbor graph.

We pass a fixed `batch_size=2048` here. `TrainConfig(batch_size='auto')` is also supported: it resolves the batch from the panel size and GPU memory and scales the learning rate by `sqrt(effective_batch / 2048)`, which is useful for throughput on large panels. Note that a very large batch gives the per-batch codebook diversity term fewer updates, so for the evenest codebook on a dataset this size a moderate fixed batch is a good choice; the resolved value is always recorded in `training_runtime.json`.

In [3]:
mc = ModelConfig(
    input_dim=adata.n_vars,
    gene_names=tuple(adata.var_names.astype(str)),
    encoder_type='mlp_deep',       # library default encoder
    quantizer_type='vq',           # recommended default quantizer
    cell_num_embeddings=256,
    neighborhood_num_embeddings=32,
)
tc = TrainConfig(
    num_epochs=30,                 # real cohort: ~300
    batch_size=2048,               # or 'auto' to resolve from panel size + GPU memory
    k_neighbors=20,
    spatial_graph='knn_radius', radius=50.0,
    neighborhood_aggregation='weighted_mean',
    save_best=False, seed=49,
)
CKPT = 'runs/quickstart'
model, adata = nv.train_model(adata, CKPT, model_config=mc, train_config=tc, sample_col='sample_id')

epoch 1/30:   0%|          | 0/56 [00:00<?, ?it/s]

epoch 2/30:   0%|          | 0/56 [00:00<?, ?it/s]

epoch 3/30:   0%|          | 0/56 [00:00<?, ?it/s]

epoch 4/30:   0%|          | 0/56 [00:00<?, ?it/s]

epoch 5/30:   0%|          | 0/56 [00:00<?, ?it/s]

...

epoch 26/30:   0%|          | 0/56 [00:00<?, ?it/s]

epoch 27/30:   0%|          | 0/56 [00:00<?, ?it/s]

epoch 28/30:   0%|          | 0/56 [00:00<?, ?it/s]

epoch 29/30:   0%|          | 0/56 [00:00<?, ?it/s]

epoch 30/30:   0%|          | 0/56 [00:00<?, ?it/s]

## What landed in the checkpoint directory

`train_model` writes the model, both codebooks, per-cell embeddings and code indices, the loss curve, a runtime record, and an annotated AnnData.

In [4]:
import os, json
print(sorted(os.listdir(CKPT)))
runtime = json.load(open(os.path.join(CKPT, 'training_runtime.json')))
print('\nruntime:', json.dumps(runtime, indent=2))

['adata_with_hierarchical_embeddings.h5ad', 'cell_codebook.npz', 'env_snapshot.json', 'hierarchical_cell_embeddings.npz', 'hierarchical_cell_indices.npz', 'hierarchical_neighborhood_embeddings.npz', 'hierarchical_neighborhood_indices.npz', 'hierarchical_vqvae_checkpoint.json', 'hierarchical_vqvae_checkpoint.pt', 'neighborhood_codebook.npz', 'train_config.json', 'training_losses.json', 'training_runtime.json']

runtime: {
  "total_seconds": 67.87,
  "total_hms": "0:01:07",
  "n_epochs": 30,
  "mean_epoch_seconds": 2.262,
  "cells_per_second": 50118.83,
  "iters_per_second": 24.753,
  "n_cells": 113385,
  "effective_batch_size": 2048,
  "n_batches_per_epoch": 56,
  "peak_gpu_gb": 0.931,
  "device": "cuda",
  "encoder_type": "mlp_plr",
  "quantizer_type": "vq",
  "input_dim": 368
}


The codes are attached back onto the AnnData: `obs['cell_codebook_idx']` (0..255), `obs['neighborhood_codebook_idx']` (0..31), and the continuous embeddings in `obsm`.

In [5]:
print(nv.anndata_keys())
cell_idx = adata.obs['cell_codebook_idx'].to_numpy()
neigh_idx = adata.obs['neighborhood_codebook_idx'].to_numpy()
print('cell codes used:', len(np.unique(cell_idx)), '/', mc.cell_num_embeddings)
print('niches used:', len(np.unique(neigh_idx)), '/', mc.neighborhood_num_embeddings)
print('embedding shapes:', adata.obsm['X_cell_embedding'].shape, adata.obsm['X_neighborhood_embedding'].shape)

{'cell_code': 'cell_codebook_idx', 'neighborhood_code': 'neighborhood_codebook_idx', 'sample': 'sample_id', 'cell_embedding': 'X_cell_embedding', 'neighborhood_embedding': 'X_neighborhood_embedding', 'spatial': 'spatial'}
cell codes used: 214 / 256
niches used: 31 / 32
embedding shapes: (113385, 64) (113385, 256)


## Per-code top markers

A quick way to read the cell codebook: for each code, the genes most enriched in cells assigned to it (mean log-normalized expression, z-scored across codes). This is a lightweight preview; the package provides a full annotation workflow (`nicheverse.annotate`) with DEGs, site distribution, and literature grounding.

In [6]:
# log-normalize a copy for the marker summary (train_model normalized internally,
# but the returned adata.X here is already log-normalized)
expr = adata.copy()
if 'log1p' not in expr.uns:
    sc.pp.normalize_total(expr); sc.pp.log1p(expr)
X = expr.X.toarray() if hasattr(expr.X, 'toarray') else np.asarray(expr.X)
genes = np.array(expr.var_names.astype(str))
codes = np.unique(cell_idx)
mean_by_code = np.vstack([X[cell_idx == k].mean(0) for k in codes])
z = (mean_by_code - mean_by_code.mean(0)) / (mean_by_code.std(0) + 1e-8)
rows = []
for i, k in enumerate(codes):
    top = genes[np.argsort(z[i])[::-1][:5]]
    rows.append({'code': int(k), 'n_cells': int((cell_idx == k).sum()), 'top_markers': ', '.join(top)})
marker_table = pd.DataFrame(rows).sort_values('n_cells', ascending=False)
marker_table.head(15)

,code,n_cells,top_markers
210,250,6454,"Tax1bp1, Dmrtb1, Hapln1, Reep6, Inadl"
126,150,2637,"Dmrtb1, Mafb, Tulp1, Lima1, Nrl"
170,200,2417,"Drd4, Nxph2, Mef2c, Ybx3, Reep6"
191,223,2340,"Neurod1, Dmrtb1, Tax1bp1, Inadl, Tulp1"
21,28,2295,"Tax1bp1, Inadl, Nr2e3, Tulp1, Reep6"
85,104,2255,"Lmo3, 4833423E24Rik, Nr2e3, Reep6, Lima1"
78,95,1813,"Neurod1, Prom1, Nr2e3, Inadl, Reep6"
156,182,1790,"Tax1bp1, Nr2e3, Reep6, Lima1, Tulp1"
212,254,1698,"Neurod1, Mfap5, Tulp1, Prokr1, Nr2e3"
70,86,1598,"Cpm, Hapln1, Nrl, Nr2e3, Prom1"


## Code-usage bar chart

How evenly the cells spread across the cell codebook. A healthy codebook uses most codes; codebook fullness increases with the biological diversity and scale of the dataset.

In [7]:
counts = pd.Series(cell_idx).value_counts().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(range(len(counts)), counts.values, width=1.0)
ax.set_xlabel('cell code (sorted by usage)'); ax.set_ylabel('n cells'); ax.set_yscale('log')
ax.set_title(f'cell codebook usage ({len(counts)}/{mc.cell_num_embeddings} codes used)')
fig.tight_layout(); plt.show()
print('done')

done


## Next steps

- **02_transcript_context.ipynb** adds the segmentation-free molecular field.
- **03_molecule_set.ipynb** uses the subcellular transcript point cloud.
- **04_apply_to_new_data.ipynb** assigns this codebook to a held-out sample.

For real cohorts set `num_epochs ~300` and annotate the codebook with `nicheverse.annotate` (per-code DEGs + literature grounding).